# Feature Engineering
The process of creating, transforming, or selecting input variables (features) from raw data to improve the performance of machine learning models. It often involves techniques like scaling, encoding categorical data, creating new features from existing ones, or reducing dimensionality. Essentially, it’s about turning raw data into meaningful inputs that help models learn patterns more effectively.

Create meaningful features:
- One-hot encoding
- Extract date features (weekday, month, hour…)
- Domain logic features
- Combine or derive features (ratios, interactions)
- Transform skew (log transform)
- Text preprocessing (tokenisation, embeddings)

Here we will be using pandas to handle storing and processing the data

In [1]:
import pandas as pd

df = pd.read_csv("data/nba_games.csv",index_col=0)

In [2]:
df.head()

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,ft%,...,tov%_max_opp,usg%_max_opp,ortg_max_opp,drtg_max_opp,team_opp,total_opp,home_opp,season,date,won
2630,240.0,39.0,75.0,0.520,13.0,19.0,0.684,22.0,22.0,1.000,...,50.0,29.7,150.0,133.0,UTA,104,0,2017,2016-10-25,True
2631,240.0,40.0,82.0,0.488,8.0,24.0,0.333,16.0,16.0,1.000,...,50.0,39.2,220.0,120.0,POR,113,1,2017,2016-10-25,False
2632,240.0,40.0,85.0,0.471,7.0,33.0,0.212,13.0,18.0,0.722,...,100.0,36.2,263.0,112.0,SAS,129,0,2017,2016-10-25,False
2633,240.0,47.0,98.0,0.480,12.0,24.0,0.500,23.0,26.0,0.885,...,75.0,31.8,139.0,143.0,GSW,100,1,2017,2016-10-25,True
2634,240.0,45.0,94.0,0.479,13.0,35.0,0.371,14.0,19.0,0.737,...,50.0,36.8,153.0,125.0,NYK,88,0,2017,2016-10-25,True


In [3]:
def remove_cols(df, cols):
    for col in cols:
        del df[col]
    return df

## Rolling averages
First useful feature to add to our dataframe is the rolling averages which our model can use to see how each team is performing over their last 10 games

In [4]:
removed_columns = ["season", "date", "won", "team", "team_opp"]
selected_columns = df.columns[~df.columns.isin(removed_columns)]

In [5]:
def find_team_averages(group):
    # Compute rolling mean with a 10-game window, shifted so current row isn’t included
    return group.shift().rolling(10, min_periods=1).mean()

# Select relevant columns
cols_to_roll = list(selected_columns) + ["won"]

# Apply rolling per team per season
df_rolling = df.groupby(["team", "season"], group_keys=False)[cols_to_roll].apply(find_team_averages)

In [6]:
df_rolling.head()

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,ft%,...,ast%_max_opp,stl%_max_opp,blk%_max_opp,tov%_max_opp,usg%_max_opp,ortg_max_opp,drtg_max_opp,total_opp,home_opp,won
2630,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2631,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2632,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2633,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2634,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
rolling_cols = [f"{col}_10" for col in df_rolling.columns]
df_rolling.columns = rolling_cols

In [8]:
df = pd.concat([df, df_rolling], axis=1)

In [9]:
df = df.dropna() #Drop all null/na values

In [10]:
df["date"].head()

2656    2016-10-27
2657    2016-10-27
2659    2016-10-27
2660    2016-10-27
2664    2016-10-28
Name: date, dtype: object

In [11]:
df.describe()

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,ft%,...,ast%_max_opp_10,stl%_max_opp_10,blk%_max_opp_10,tov%_max_opp_10,usg%_max_opp_10,ortg_max_opp_10,drtg_max_opp_10,total_opp_10,home_opp_10,won_10
count,7758.000000,7758.000000,7758.000000,7758.000000,7758.000000,7758.000000,7758.000000,7758.000000,7758.000000,7758.000000,...,7758.000000,7758.000000,7758.000000,7758.000000,7758.000000,7758.000000,7758.000000,7758.000000,7758.000000,7758.000000
mean,241.559680,39.815545,86.745424,0.459925,10.533643,29.447151,0.357366,17.426399,22.663057,0.769257,...,42.527850,5.853619,8.707056,39.750953,34.669270,176.396435,117.995141,107.468484,0.499350,0.508478
std,7.213461,5.178010,7.308698,0.054150,3.657200,7.184217,0.090347,6.013344,7.288129,0.102584,...,5.243298,1.452140,2.353236,7.703331,2.938745,13.780902,4.956999,6.224827,0.145607,0.215967
min,240.000000,21.000000,61.000000,0.278000,1.000000,7.000000,0.053000,1.000000,1.000000,0.143000,...,22.200000,2.100000,0.000000,16.100000,24.800000,109.000000,87.000000,76.000000,0.000000,0.000000
25%,240.000000,36.000000,82.000000,0.422000,8.000000,24.000000,0.296000,13.000000,18.000000,0.706000,...,38.910000,4.970000,7.150000,34.310000,32.670000,166.900000,114.800000,103.100000,0.400000,0.400000
50%,240.000000,40.000000,86.000000,0.459000,10.000000,29.000000,0.356000,17.000000,22.000000,0.778000,...,42.215000,5.590000,8.300000,39.040000,34.200000,175.800000,117.900000,107.354167,0.500000,0.500000
75%,240.000000,43.000000,91.000000,0.494000,13.000000,34.000000,0.417000,21.000000,27.000000,0.840000,...,45.807500,6.418125,9.869167,44.250000,36.100000,185.100000,121.400000,111.600000,0.600000,0.700000
max,340.000000,61.000000,128.000000,0.667000,27.000000,70.000000,0.842000,44.000000,64.000000,1.000000,...,100.000000,23.000000,26.560000,100.000000,71.200000,300.000000,143.000000,140.000000,1.000000,1.000000


### Adding days rested since last game

In [12]:
def set_days_rested(df,opponent=False):
    addition = ""
    if opponent: addition = "_opp"

    team = "team"+addition
    prev_date = "prev_date"+addition
    days_rested = "days_rested"+addition
    
    # Sort within each team by date
    df = df.sort_values([team, "date"])

    # Compute previous game date for each team
    df[prev_date] = df.groupby(team)["date"].shift(1)
    
    #Set the dates as datetime
    df["date"] = pd.to_datetime(df["date"])
    df[prev_date] = pd.to_datetime(df[prev_date])

    # Difference in days
    df[days_rested] = (df["date"] - df[prev_date]).dt.days

    # First game of the season has no previous game → fill with something reasonable
    df[days_rested] = df[days_rested].fillna(0)

    return df

df = set_days_rested(df)
df = set_days_rested(df,True)

In [13]:
df = df.dropna()
df = df.sort_values("date")

In [14]:
df = df[df["days_rested"] <= 4]
df = df[df["days_rested_opp"] <= 4]

In [15]:
df[["team","team_opp","home","season","days_rested","days_rested_opp"]].head()

,team,team_opp,home,season,days_rested,days_rested_opp
2696,IND,CHI,0,2017,1.0,2.0
2681,NOP,SAS,0,2017,1.0,2.0
2694,BOS,CHO,0,2017,2.0,1.0
2692,ORL,CLE,0,2017,1.0,1.0
2688,SAS,NOP,1,2017,2.0,1.0


In [16]:
df.head()

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,ft%,...,usg%_max_opp_10,ortg_max_opp_10,drtg_max_opp_10,total_opp_10,home_opp_10,won_10,prev_date,days_rested,prev_date_opp,days_rested_opp
2696,240.0,43.0,88.0,0.489,7.0,18.0,0.389,8.0,9.0,0.889,...,33.75,189.5,111.0,112.0,0.5,0.5,2016-10-28,1.0,2016-10-27,2.0
2681,240.0,32.0,86.0,0.372,4.0,22.0,0.182,11.0,15.0,0.733,...,27.50,225.0,107.5,114.5,0.0,0.0,2016-10-28,1.0,2016-10-27,2.0
2694,240.0,39.0,81.0,0.481,15.0,31.0,0.484,11.0,17.0,0.647,...,30.65,196.5,117.5,111.0,0.5,0.5,2016-10-27,2.0,2016-10-28,1.0
2692,240.0,38.0,84.0,0.452,8.0,28.0,0.286,15.0,26.0,0.577,...,31.65,188.0,105.0,108.0,0.5,0.0,2016-10-28,1.0,2016-10-28,1.0
2688,240.0,35.0,83.0,0.422,10.0,24.0,0.417,18.0,23.0,0.783,...,36.20,131.5,131.5,97.0,1.0,1.0,2016-10-27,2.0,2016-10-28,1.0


In [17]:
df[df["season"] == 2017].count()

mp                 2491
fg                 2491
fga                2491
fg%                2491
3p                 2491
                   ... 
won_10             2491
prev_date          2491
days_rested        2491
prev_date_opp      2491
days_rested_opp    2491
Length: 286, dtype: int64

In [18]:
for col in df.columns:
    print(col)

mp
fg
fga
fg%
3p
3pa
3p%
ft
fta
ft%
orb
drb
trb
ast
stl
blk
tov
pf
pts
ts%
efg%
3par
ftr
orb%
drb%
trb%
ast%
stl%
blk%
tov%
usg%
ortg
drtg
fg_max
fga_max
fg%_max
3p_max
3pa_max
3p%_max
ft_max
fta_max
ft%_max
orb_max
drb_max
trb_max
ast_max
stl_max
blk_max
tov_max
pf_max
pts_max
gmsc_max
+/-_max
ts%_max
efg%_max
3par_max
ftr_max
orb%_max
drb%_max
trb%_max
ast%_max
stl%_max
blk%_max
tov%_max
usg%_max
ortg_max
drtg_max
team
total
home
mp_opp
fg_opp
fga_opp
fg%_opp
3p_opp
3pa_opp
3p%_opp
ft_opp
fta_opp
ft%_opp
orb_opp
drb_opp
trb_opp
ast_opp
stl_opp
blk_opp
tov_opp
pf_opp
pts_opp
ts%_opp
efg%_opp
3par_opp
ftr_opp
orb%_opp
drb%_opp
trb%_opp
ast%_opp
stl%_opp
blk%_opp
tov%_opp
usg%_opp
ortg_opp
drtg_opp
fg_max_opp
fga_max_opp
fg%_max_opp
3p_max_opp
3pa_max_opp
3p%_max_opp
ft_max_opp
fta_max_opp
ft%_max_opp
orb_max_opp
drb_max_opp
trb_max_opp
ast_max_opp
stl_max_opp
blk_max_opp
tov_max_opp
pf_max_opp
pts_max_opp
gmsc_max_opp
+/-_max_opp
ts%_max_opp
efg%_max_opp
3par_max_opp
ftr_max_opp
orb%_m

### Balancing the dataset 

In [19]:
print(len(df[df['won'] == 1]))
print(len(df[df['won'] == 0]))

3734
3733


In [20]:
from sklearn.utils import resample

def balance_dataframe(df):
    wins = df[df['won'] == 1]
    losses = df[df['won'] == 0]
    
    if len(wins) == len(losses): return df
    if len(wins) > len(losses):
        wins_down = resample(wins,
                   replace=False,
                   n_samples=len(losses),
                   random_state=42)

        return pd.concat([losses, wins_down])
    
    losses_down = resample(losses,
               replace=False,
               n_samples=len(wins),
               random_state=42)
    return pd.concat([wins, losses_down])

df = balance_dataframe(df)

In [21]:
print(len(df[df['won'] == 1]))
print(len(df[df['won'] == 0]))

3733
3733


# Final Dataframe
The final features must only include past data and current information, that can both be inputted into the model.

For the ai model to predict which team is going to win, it needs the recent stats from each team playing.

### The primary key used for finding the most recent game played and pull statistics from that game
- season
- date
- team (home team)
- team_opp (away team)

### Information the model uses to make predictions
- rolling 10-game averages team (contains '_10')
- rolling 10-game averages team_opp (contains '_opp_10')
- rolling 5-game averages team (contains '_5')
- rolling 5-game averages team_opp (contains '_opp_5')
- team performance against team_opp in past 3 games
- team_opp performance against team in past 3 games
- days rested team
- days rested team_opp

### Label/Outcome/Loss/Class
- won   <-- used as label, test model performance with this feature

In [22]:
cols_10 = [col for col in df.columns if "_10" in col]
selected_columns = ["season","date","team","team_opp"] + cols_10 + ["days_rested","days_rested_opp"] + ["won"]

In [23]:
for column in selected_columns:
    print(column)

season
date
team
team_opp
mp_10
fg_10
fga_10
fg%_10
3p_10
3pa_10
3p%_10
ft_10
fta_10
ft%_10
orb_10
drb_10
trb_10
ast_10
stl_10
blk_10
tov_10
pf_10
pts_10
ts%_10
efg%_10
3par_10
ftr_10
orb%_10
drb%_10
trb%_10
ast%_10
stl%_10
blk%_10
tov%_10
usg%_10
ortg_10
drtg_10
fg_max_10
fga_max_10
fg%_max_10
3p_max_10
3pa_max_10
3p%_max_10
ft_max_10
fta_max_10
ft%_max_10
orb_max_10
drb_max_10
trb_max_10
ast_max_10
stl_max_10
blk_max_10
tov_max_10
pf_max_10
pts_max_10
gmsc_max_10
+/-_max_10
ts%_max_10
efg%_max_10
3par_max_10
ftr_max_10
orb%_max_10
drb%_max_10
trb%_max_10
ast%_max_10
stl%_max_10
blk%_max_10
tov%_max_10
usg%_max_10
ortg_max_10
drtg_max_10
total_10
home_10
mp_opp_10
fg_opp_10
fga_opp_10
fg%_opp_10
3p_opp_10
3pa_opp_10
3p%_opp_10
ft_opp_10
fta_opp_10
ft%_opp_10
orb_opp_10
drb_opp_10
trb_opp_10
ast_opp_10
stl_opp_10
blk_opp_10
tov_opp_10
pf_opp_10
pts_opp_10
ts%_opp_10
efg%_opp_10
3par_opp_10
ftr_opp_10
orb%_opp_10
drb%_opp_10
trb%_opp_10
ast%_opp_10
stl%_opp_10
blk%_opp_10
tov%_opp_10
us

In [24]:
df[selected_columns].head()

,season,date,team,team_opp,mp_10,fg_10,fga_10,fg%_10,3p_10,3pa_10,...,tov%_max_opp_10,usg%_max_opp_10,ortg_max_opp_10,drtg_max_opp_10,total_opp_10,home_opp_10,won_10,days_rested,days_rested_opp,won
2696,2017,2016-10-29,IND,CHI,252.5,42.0,95.5,0.4415,8.5,21.5,...,25.00,33.75,189.5,111.0,112.0,0.5,0.5,1.0,2.0,False
2681,2017,2016-10-29,NOP,SAS,240.0,42.5,96.0,0.4415,4.0,20.5,...,34.70,27.50,225.0,107.5,114.5,0.0,0.0,1.0,2.0,False
2692,2017,2016-10-29,ORL,CLE,240.0,33.5,92.0,0.3645,7.0,22.0,...,58.05,31.65,188.0,105.0,108.0,0.5,0.0,1.0,1.0,False
2709,2017,2016-10-30,DAL,HOU,252.5,40.0,92.0,0.4355,13.0,38.5,...,31.25,38.20,167.5,114.0,118.0,0.5,0.0,2.0,2.0,False
2702,2017,2016-10-30,LAL,OKC,240.0,38.5,86.5,0.4435,10.5,29.0,...,65.00,37.95,176.5,118.0,105.0,0.5,0.5,2.0,2.0,False


In [25]:
df = df[selected_columns]

In [26]:
df.to_pickle('df_nba_games.pkl')